# Context-aware trial chunks

In [1]:
from pathlib import Path
from statistics import mean
import re

import chromadb
import pandas as pd
from IPython.display import display
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm.auto import tqdm
from transformers import AutoTokenizer


/home/yan/miniconda3/envs/LocalAgent_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
md_dir = Path("/home/yan/test/RAG/demo/data/trec_pm2019/trial_markdown")
db_dir = Path("/home/yan/test/RAG/chroma_context_db")
collection_name = "trec_2019_trials_context"
model_name = "BAAI/bge-small-en-v1.5"
device = "cuda"
max_tokens = 450
chunk_overlap = 50
background_limits = {"Title": 60, "Conditions": 60, "Interventions": 40}
body_sections = ("Brief Summary", "Detailed Description", "Eligibility Criteria")

## Build chunks

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
if max_tokens > tokenizer.model_max_length:
    raise ValueError("Chunk size exceeds the tokenizer limit")


def read_sections(text):
    parts = re.split(r"(?m)^##[ \t]+([^\n]+)\n", text)
    return {parts[i].strip(): parts[i + 1].strip()
            for i in range(1, len(parts), 2)}


def clip_field(text, limit):
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    if len(encoded["input_ids"]) <= limit:
        return text, False
    end = encoded["offset_mapping"][limit - 1][1]
    return text[:end].rstrip(), True


chunks, chunk_ids, token_lengths = [], [], []
truncated_trials, fallback_trials = set(), []
files = sorted(md_dir.glob("NCT*.md"))
if not files:
    raise ValueError(f"No trial files in {md_dir}")

for path in tqdm(files, desc="Building context chunks"):
    sections = read_sections(path.read_text(encoding="utf-8"))
    background = []
    for field, limit in background_limits.items():
        value, clipped = clip_field(sections.get(field, ""), limit)
        if value:
            background.append(f"{field}: {value}")
        if clipped:
            truncated_trials.add(path.stem)
    context = "\n".join(background)
    bodies = [(name, sections[name]) for name in body_sections if sections.get(name)]
    if not bodies:
        fallback_trials.append(path.stem)
        bodies = [("Overview", "\n".join(
            f"{name}: {value}" for name, value in sections.items()
            if name not in background_limits and name != "NCT ID" and value
        ) or "No narrative section provided.")]
    for section_name, body in bodies:
        prefix = context + f"\n\nSection: {section_name}\n"
        body_budget = max_tokens - len(tokenizer.encode(prefix)) - 8
        if body_budget <= chunk_overlap:
            raise ValueError(f"Insufficient body budget: {path.stem}")
        splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer, separators=["\n\n", "\n", ". ", "; ", " ", ""],
            chunk_size=body_budget, chunk_overlap=chunk_overlap,
        )
        for index, part in enumerate(splitter.split_text(body)):
            text = prefix + part
            length = len(tokenizer.encode(text))
            if length > max_tokens:
                raise ValueError(f"Oversized chunk: {path.stem}, {section_name}, {length}")
            chunks.append(Document(page_content=text, metadata={
                "nct_id": path.stem,
                "source": f"data/trec_pm2019/trial_markdown/{path.name}",
                "dataset": "TREC PM 2019", "section": section_name,
                "background_truncated": path.stem in truncated_trials,
            }))
            chunk_ids.append(f"{path.stem}:{section_name}:{index}")
            token_lengths.append(length)
assert {p.stem for p in files} == {c.metadata["nct_id"] for c in chunks}
assert len(chunk_ids) == len(set(chunk_ids))
chunking_stats = {
    "documents": len(files), "chunks": len(chunks),
    "maximum_tokens": max(token_lengths), "mean_tokens": mean(token_lengths),
    "trials_with_truncated_background": len(truncated_trials),
    "trials_without_narrative_sections": len(fallback_trials),
    "background_field_token_limits": background_limits,
    "body_sections": list(body_sections), "overlap": chunk_overlap,
}
display(pd.DataFrame([chunking_stats]))
print(chunks[0].page_content)

Building context chunks:   0%|          | 0/8567 [00:00<?, ?it/s]

Building context chunks:   0%|          | 9/8567 [00:00<01:55, 74.12it/s]

Building context chunks:   0%|          | 18/8567 [00:00<01:47, 79.21it/s]

Building context chunks:   0%|          | 26/8567 [00:00<02:19, 61.05it/s]

Building context chunks:   0%|          | 33/8567 [00:00<02:26, 58.36it/s]

...

Building context chunks: 100%|█████████▉| 8563/8567 [03:26<00:00, 35.78it/s]

Building context chunks: 100%|██████████| 8567/8567 [03:26<00:00, 36.16it/s]

Building context chunks: 100%|██████████| 8567/8567 [03:26<00:00, 41.54it/s]

,documents,chunks,maximum_tokens,mean_tokens,trials_with_truncated_background,trials_without_narrative_sections,background_field_token_limits,body_sections,overlap
0,8567,43459,442,302.000276,762,0,"{'Title': 60, 'Conditions': 60, 'Interventions...","[Brief Summary, Detailed Description, Eligibil...",50


Title: 41.8 Degree Centigrade Whole Body Hyperthermia for the Treatment of Rheumatoid Diseases
Conditions: Rheumatic Diseases
Interventions: Whole body hyperthermia unit

Section: Brief Summary
Recently a non-toxic system for whole body hyperthermia (WBH) used at the University of Wisconsin has been shown to induce soluble tumor necrosis factor-receptor (sTNF-R) I and II when patients are heated systemically to 41.8C for 60 minutes. This observation might provide a biological basis for the therapeutic application of WBH to rheumatoid diseases, for which there is a positive anecdotal clinical experience. Inherent in the hypothesis which is the basis for this protocol is the concept that the induction of TNF receptors by WBH may induce a remission in patients with active rheumatoid arthritis. Beyond clinical response the biological endpoint for this investigation includes cytokine levels, TNF levels, sTNF-R levels and changes in cellular TNF receptors.


## Index a new collection

In [4]:
client = chromadb.PersistentClient(path=str(db_dir.resolve()))
existing_names = {c.name for c in client.list_collections()}
if collection_name in existing_names:
    raise ValueError("Collection already exists. Use a new collection name to rebuild.")
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": device, "local_files_only": True},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)
vectorstore = Chroma(
    collection_name=collection_name, client=client,
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)
for start in tqdm(range(0, len(chunks), 256), desc="Indexing context chunks"):
    vectorstore.add_documents(
        documents=chunks[start:start + 256], ids=chunk_ids[start:start + 256]
    )
collection = client.get_collection(collection_name)
assert collection.count() == len(chunks)
corpus_ids = {chunk.metadata["nct_id"] for chunk in chunks}
print("Stored chunks:", collection.count())
print("Trials:", len(corpus_ids))
print("Index configuration:", collection.configuration)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  52%|█████▏    | 104/199 [00:00<00:00, 995.32it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1446.59it/s]

Indexing context chunks:   0%|          | 0/170 [00:00<?, ?it/s]

Indexing context chunks:   1%|          | 1/170 [00:02<05:39,  2.01s/it]

...

Indexing context chunks:  99%|█████████▉| 168/170 [03:41<00:03,  1.67s/it]

Indexing context chunks:  99%|█████████▉| 169/170 [03:42<00:01,  1.55s/it]

Indexing context chunks: 100%|██████████| 170/170 [03:43<00:00,  1.40s/it]

Indexing context chunks: 100%|██████████| 170/170 [03:43<00:00,  1.32s/it]

Stored chunks: 43459
Trials: 8567
Index configuration: {'hnsw': {'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': None}


## Read source Markdown

In [ ]:
trial_paths = {path.stem: path for path in md_dir.glob("NCT*.md")}

nct_id = next(iter(trial_paths))
trial_text = trial_paths[nct_id].read_text(encoding="utf-8")
